In [3]:
import psycopg2
import pandas as pd

conn = psycopg2.connect(
    dbname="steam",
    user="kesiyun",
    host="localhost"
)

pd.read_sql(
    "SELECT COUNT(*) FROM items",
    conn
)

/var/folders/dn/8xt3xqsn3ysf586z6l0zvfym0000gn/T/ipykernel_7226/3014129503.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(


,count
0,25899


In [2]:
!pip install psycopg2-binary

     |████████████████████████████████| 3.0 MB 1.8 MB/s eta 0:00:01


# database check:

In [5]:
pd.read_sql("""
SELECT COUNT(*)
FROM item_embeddings
""", conn)

/var/folders/dn/8xt3xqsn3ysf586z6l0zvfym0000gn/T/ipykernel_4440/1836850851.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql("""


,count
0,25899


In [6]:
target_id = 413150

sql = """
SELECT
    i.item_id,
    i.name,
    ROUND(
        (1 - (e.embedding <=> s.embedding))::numeric,
        4
    ) AS similarity
FROM item_embeddings s
JOIN item_embeddings e
    ON e.item_id <> s.item_id
JOIN items i
    ON i.item_id = e.item_id
WHERE s.item_id = %s
ORDER BY e.embedding <=> s.embedding
LIMIT 10;
"""

df = pd.read_sql(
    sql,
    conn,
    params=(target_id,)
)

df

/var/folders/dn/8xt3xqsn3ysf586z6l0zvfym0000gn/T/ipykernel_4440/3193981562.py:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


,item_id,name,similarity
0,1137750,Farmer's Life,0.7391
1,1156360,Peaceful Days,0.7247
2,751780,Forager,0.7222
3,4484350,Hatarake! - Working Hard! Even though I gettin...,0.7217
4,2111170,STORY OF SEASONS: A Wonderful Life,0.7172
5,3057760,CROPS!,0.7169
6,1185820,Hollow Island,0.7143
7,1062520,Dinkum,0.7042
8,1392960,STORY OF SEASONS: Pioneers of Olive Town,0.7005
9,2569670,Farmer's Dynasty 2,0.6962


In [7]:
def recommend_similar_vector(
    item_id,
    top_k=10
):
    sql = """
    SELECT
        i.item_id,
        i.name,
        ROUND(
            (1 - (e.embedding <=> s.embedding))::numeric,
            4
        ) AS similarity
    FROM item_embeddings s
    JOIN item_embeddings e
      ON e.item_id <> s.item_id
    JOIN items i
      ON i.item_id = e.item_id
    WHERE s.item_id = %s
    ORDER BY e.embedding <=> s.embedding
    LIMIT %s
    """

    return pd.read_sql(
        sql,
        conn,
        params=(item_id, top_k)
    )

In [10]:
recommend_similar_vector(413150)


/var/folders/dn/8xt3xqsn3ysf586z6l0zvfym0000gn/T/ipykernel_4440/32876988.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(


,item_id,name,similarity
0,1137750,Farmer's Life,0.7391
1,1156360,Peaceful Days,0.7247
2,751780,Forager,0.7222
3,4484350,Hatarake! - Working Hard! Even though I gettin...,0.7217
4,2111170,STORY OF SEASONS: A Wonderful Life,0.7172
5,3057760,CROPS!,0.7169
6,1185820,Hollow Island,0.7143
7,1062520,Dinkum,0.7042
8,1392960,STORY OF SEASONS: Pioneers of Olive Town,0.7005
9,2569670,Farmer's Dynasty 2,0.6962


In [11]:
recommend_similar_vector(227300)

/var/folders/dn/8xt3xqsn3ysf586z6l0zvfym0000gn/T/ipykernel_4440/32876988.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(


,item_id,name,similarity
0,873840,Truck & Logistics Simulator,0.7307
1,768180,Truck Driver,0.7094
2,3811420,Transit King Tycoon,0.6885
3,995680,Truck Mechanic: Dangerous Paths,0.6680
4,232010,Euro Truck Simulator,0.6596
5,1084020,TheoTown,0.6504
6,1066780,Transport Fever 2,0.6412
7,4252340,CPM 2,0.6383
8,3325500,Worldwide Rush,0.6363
9,936500,Transport INC,0.6349


# initial searches
* Free text -> N games (keyword)
* game id -> N games (vector)

In [15]:
import psycopg2
import pandas as pd

conn = psycopg2.connect(
    dbname="steam",
    user="kesiyun",
    host="localhost"
)

In [16]:
def print_row(row):

    print("-" * 80)

    print(
        f"{row['name']} "
        f"(id={row['item_id']}) "
        f"score={row['score']:.4f}"
    )

    print()

    print(
        row["summary"][:200]
    )

In [43]:
import psycopg2
import pandas as pd

conn = psycopg2.connect(
    dbname="steam",
    user="kesiyun",
    host="localhost"
)


def print_game(game, score):

    print("-" * 80)

    print(
        f"{game['name']} "
        f"(id={game['item_id']}) "
        f"score={score:.4f}"
    )

    tags = game.get("tags", {})

    if tags.get("mood"):
        print(
            "Mood:",
            ", ".join(tags["mood"])
        )

    if tags.get("theme"):
        print(
            "Theme:",
            ", ".join(tags["theme"])
        )

    print(
        "Summary:",
        game["summary"][:180]
    )


def search_db(
    query,
    top_k=10
):

    sql = """
    SELECT
        i.item_id,
        i.name,
        a.summary,
        a.tags,

        ts_rank_cd(
            to_tsvector(
                'english',
                coalesce(i.description_clean,'')
                || ' ' ||
                coalesce(a.summary,'')
                || ' ' ||
                coalesce(a.tags::text,'')
            ),
            websearch_to_tsquery(
                'english',
                %s
            )
        ) AS score

    FROM items i
    JOIN item_annotations a
        USING(item_id)

    ORDER BY score DESC

    LIMIT %s
    """

    df = pd.read_sql(
        sql,
        conn,
        params=(
            query,
            top_k
        )
    )

    results = []

    for _, row in df.iterrows():

        results.append(
            (
                {
                    "item_id": row["item_id"],
                    "name": row["name"],
                    "summary": row["summary"],
                    "tags": row["tags"]
                },
                float(row["score"])
            )
        )

    return results


def recommend_similar_db(
    item_id,
    top_k=10
):

    sql = """
    SELECT
        i.item_id,
        i.name,
        a.summary,
        a.tags,

        ROUND(
            (
                1 -
                (
                    e.embedding
                    <=>
                    s.embedding
                )
            )::numeric,
            4
        ) AS score

    FROM item_embeddings s

    JOIN item_embeddings e
        ON e.item_id <> s.item_id

    JOIN items i
        ON i.item_id = e.item_id

    JOIN item_annotations a
        ON a.item_id = i.item_id

    WHERE s.item_id = %s

    ORDER BY
        e.embedding
        <=>
        s.embedding

    LIMIT %s
    """

    df = pd.read_sql(
        sql,
        conn,
        params=(
            item_id,
            top_k
        )
    )

    results = []

    for _, row in df.iterrows():

        results.append(
            (
                {
                    "item_id": row["item_id"],
                    "name": row["name"],
                    "summary": row["summary"],
                    "tags": row["tags"]
                },
                float(row["score"])
            )
        )

    return results

In [45]:
# ============================================================
# DEMO 1
# ============================================================
#given a free-text query (e.g., "a relaxing video game with beautiful art but no combat"), return a ranked list of items by combining keyword matching (e.g., BM25) with vector similarity search.

print("\n")
print("=" * 80)
print("FREE TEXT RETRIEVAL")
print("=" * 80)

#query = "relaxing farming game"
query = 'A Game About Chopping Trees'
#query = 'a truck driving game featured in europe'
#query = "a relaxing video game with beautiful art but no combat"
#query = "truck driving game"

print(f"Query: {query}")

results = search_db(
    query,
    top_k=10
)

for rank, (game, score) in enumerate(
    results,
    start=1
):

    print(f"\nRank #{rank}")

    print_game(
        game,
        score
    )



FREE TEXT RETRIEVAL
Query: A Game About Chopping Trees


/var/folders/dn/8xt3xqsn3ysf586z6l0zvfym0000gn/T/ipykernel_4440/666004318.py:77: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(



Rank #1
--------------------------------------------------------------------------------
A Game About Chopping Trees (id=4512570) score=0.0818
Mood: Relaxing, Cozy, Contemplative
Theme: Nature, Slice of Life
Summary: The player lives the quiet life of a woodsman, chopping trees at their own pace, riding a handcar between peaceful forests, spending earnings on sharper axes and stronger swings, a

Rank #2
--------------------------------------------------------------------------------
Chop Chains (id=4139720) score=0.0695
Mood: Relaxing, Cheerful
Theme: Nature
Summary: A relaxing incremental game where the player hovers the cursor over trees to chop them automatically, collects wood of varying rarity, spends it on permanent skill tree upgrades, u

Rank #3
--------------------------------------------------------------------------------
Timberman VR (id=1887260) score=0.0683
Mood: Challenging, Cheerful
Theme: Nature
Summary: A retro arcade-style VR game where the player chops down trees w

In [49]:
# ============================================================
# DEMO 2
# ============================================================
# Given an item ID, retrieve the top-N most similar items

print("\n")
print("="*80)
print("ITEM TO ITEM RETRIEVAL")
print("="*80)

target_id = 227300 # ets2
#target_id = 413150 # Stardew Valley

top_n = 3

print(f"target_id: {target_id}")
print(f"top_n: {top_n}")

target = pd.read_sql(
    """
    SELECT
        item_id,
        name
    FROM items
    WHERE item_id = %s
    """,
    conn,
    params=(target_id,)
).iloc[0]

print(
    f"\nTarget item: "
    f"{target['name']} "
    f"({target_id})"
)

results = recommend_similar_db(
    target_id,
    top_k=top_n
)

for rank, (game, score) in enumerate(
    results,
    start=1
):

    print(f"\nRank #{rank}")

    print_game(
        game,
        score
    )



ITEM TO ITEM RETRIEVAL
target_id: 227300
top_n: 3

Target item: Euro Truck Simulator 2 (227300)

Rank #1
--------------------------------------------------------------------------------
Truck & Logistics Simulator (id=873840) score=0.7307
Mood: Relaxing, Challenging
Theme: Modern
Summary: A delivery simulation where the player loads pallets, containers, and vehicles with forklifts and handlers, then drives over 30 customizable vehicles across a 28 km² open world, co

Rank #2
--------------------------------------------------------------------------------
Truck Driver (id=768180) score=0.7094
Mood: Relaxing, Cheerful
Theme: Modern
Summary: After inheriting a truck from his father, the player moves to a new city and builds a trucking career, taking jobs from locals such as a constructor and a lumberjack, driving a sea

Rank #3
--------------------------------------------------------------------------------
Transit King Tycoon (id=3811420) score=0.6885
Mood: Relaxing
Theme: Modern
Summa

/var/folders/dn/8xt3xqsn3ysf586z6l0zvfym0000gn/T/ipykernel_4440/2189604653.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  target = pd.read_sql(
/var/folders/dn/8xt3xqsn3ysf586z6l0zvfym0000gn/T/ipykernel_4440/666004318.py:150: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


# hybrid searches
* Free text -> N games (keyword+vector)
* game id -> N games (keyword+vector)

In [1]:
# ============================================================
# CONFIG
# ============================================================

import os

# ----------------------------
# PostgreSQL
# ----------------------------

DB_NAME = "steam"
DB_USER = "kesiyun"
DB_HOST = "localhost"

# ----------------------------
# Embeddings
# ----------------------------

EMBEDDING_MODEL = "text-embedding-v4"
EMBEDDING_DIM = 256

# ----------------------------
# LLM
# ----------------------------

LLM_MODEL = "qwen3.8-27b"

# ----------------------------
# Hybrid Retrieval
# ----------------------------

RRF_K = 60

KEYWORD_CANDIDATES = 50
VECTOR_CANDIDATES = 50

# ----------------------------
# Default Demo Values
# ----------------------------

DEFAULT_TOP_N = 10

DEFAULT_GAME_ID = 227300
#DEFAULT_GAME_ID = 413150

DEFAULT_QUERY = (
    "a relaxing video game "
    "with beautiful art "
    "but no combat"
)

# ----------------------------
# DashScope
# ----------------------------

DASHSCOPE_API_KEY = os.getenv(
    "DASHSCOPE_API_KEY"
)

In [2]:
# ============================================================
# IMPORTS
# ============================================================

import json

import psycopg2
import pandas as pd
import dashscope

from dashscope import TextEmbedding
from collections import defaultdict


# ============================================================
# DATABASE CONNECTION
# ============================================================

conn = psycopg2.connect(
    dbname=DB_NAME,
    user=DB_USER,
    host=DB_HOST
)


# ============================================================
# HELPERS
# ============================================================

def print_game(game, score):

    print("-" * 80)

    print(
        f"{game['name']} "
        f"(id={game['item_id']}) "
        f"score={score:.4f}"
    )

    tags = game.get("tags", {})

    if tags.get("mood"):
        print(
            "Mood:",
            ", ".join(tags["mood"])
        )

    if tags.get("theme"):
        print(
            "Theme:",
            ", ".join(tags["theme"])
        )

    print(
        "Summary:",
        game["summary"][:180]
    )


# ============================================================
# KEYWORD SEARCH
# ============================================================

def keyword_search(
    query,
    top_k=KEYWORD_CANDIDATES
):

    sql = """
    SELECT
        i.item_id,
        i.name,
        a.summary,
        a.tags,

        ts_rank_cd(
            to_tsvector(
                'english',
                coalesce(i.description_clean,'')
                || ' ' ||
                coalesce(a.summary,'')
                || ' ' ||
                coalesce(a.tags::text,'')
            ),
            websearch_to_tsquery(
                'english',
                %s
            )
        ) AS score

    FROM items i
    JOIN item_annotations a
        USING(item_id)

    ORDER BY score DESC

    LIMIT %s
    """

    df = pd.read_sql(
        sql,
        conn,
        params=(query, top_k)
    )

    results = []

    for _, row in df.iterrows():

        results.append(
            (
                {
                    "item_id": row["item_id"],
                    "name": row["name"],
                    "summary": row["summary"],
                    "tags": row["tags"]
                },
                float(row["score"])
            )
        )

    return results


# ============================================================
# QUERY -> VECTOR SEARCH
# ============================================================

def vector_search(
    query,
    top_k=VECTOR_CANDIDATES
):

    response = TextEmbedding.call(
        model=EMBEDDING_MODEL,
        input=query,
        dimension=EMBEDDING_DIM
    )

    query_vector = (
        response.output["embeddings"][0]
        ["embedding"]
    )

    vector_str = (
        "["
        + ",".join(
            map(str, query_vector)
        )
        + "]"
    )

    sql = """
    SELECT
        i.item_id,
        i.name,
        a.summary,
        a.tags,

        ROUND(
            (
                1 -
                (
                    e.embedding
                    <=>
                    %s::vector
                )
            )::numeric,
            4
        ) AS score

    FROM item_embeddings e

    JOIN items i
        ON i.item_id = e.item_id

    JOIN item_annotations a
        ON a.item_id = e.item_id

    ORDER BY
        e.embedding
        <=>
        %s::vector

    LIMIT %s
    """

    df = pd.read_sql(
        sql,
        conn,
        params=(
            vector_str,
            vector_str,
            top_k
        )
    )

    results = []

    for _, row in df.iterrows():

        results.append(
            (
                {
                    "item_id": row["item_id"],
                    "name": row["name"],
                    "summary": row["summary"],
                    "tags": row["tags"]
                },
                float(row["score"])
            )
        )

    return results


# ============================================================
# ITEM -> VECTOR SEARCH
# ============================================================

def recommend_similar_db(
    item_id,
    top_k=VECTOR_CANDIDATES
):

    sql = """
    SELECT
        i.item_id,
        i.name,
        a.summary,
        a.tags,

        ROUND(
            (
                1 -
                (
                    e.embedding
                    <=>
                    s.embedding
                )
            )::numeric,
            4
        ) AS score

    FROM item_embeddings s

    JOIN item_embeddings e
        ON e.item_id <> s.item_id

    JOIN items i
        ON i.item_id = e.item_id

    JOIN item_annotations a
        ON a.item_id = e.item_id

    WHERE s.item_id = %s

    ORDER BY
        e.embedding
        <=>
        s.embedding

    LIMIT %s
    """

    df = pd.read_sql(
        sql,
        conn,
        params=(
            item_id,
            top_k
        )
    )

    results = []

    for _, row in df.iterrows():

        results.append(
            (
                {
                    "item_id": row["item_id"],
                    "name": row["name"],
                    "summary": row["summary"],
                    "tags": row["tags"]
                },
                float(row["score"])
            )
        )

    return results


# ============================================================
# HYBRID ITEM SEARCH
# ============================================================

def hybrid_similar_db(
    item_id,
    top_k=DEFAULT_TOP_N,
    candidate_k=50
):

    target = pd.read_sql(
        """
        SELECT
            i.name,
            a.summary,
            a.tags
        FROM items i
        JOIN item_annotations a
            USING(item_id)
        WHERE i.item_id = %s
        """,
        conn,
        params=(item_id,)
    ).iloc[0]

    query_text = " ".join(
        [
            target["name"],
            target["summary"],
            str(target["tags"])
        ]
    )

    keyword_results = keyword_search(
        query_text,
        top_k=candidate_k
    )

    vector_results = recommend_similar_db(
        item_id,
        top_k=candidate_k
    )

    scores = defaultdict(float)

    for rank, (game, _) in enumerate(
        keyword_results,
        start=1
    ):
        scores[game["item_id"]] += (
            1 / (RRF_K + rank)
        )

    for rank, (game, _) in enumerate(
        vector_results,
        start=1
    ):
        scores[game["item_id"]] += (
            1 / (RRF_K + rank)
        )

    game_lookup = {}

    for game, _ in keyword_results:
        game_lookup[game["item_id"]] = game

    for game, _ in vector_results:
        game_lookup[game["item_id"]] = game

    if item_id in scores:
        del scores[item_id]

    final_results = []

    for gid, score in sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )[:top_k]:

        final_results.append(
            (
                game_lookup[gid],
                score
            )
        )

    return final_results


# ============================================================
# LLM EXPLANATIONS
# ============================================================

def explain_query_match(
    query,
    game
):

    SYSTEM_PROMPT = """
You are a game recommendation explanation assistant.

Explain why the retrieved game matches the user's query.

Rules:
- Maximum 60 words.
- Friendly and natural.
- Focus on gameplay, theme and mood.
"""

    USER_PROMPT = f"""
User query:
{query}

Retrieved game:
{game['name']}

Summary:
{game['summary']}

Tags:
{json.dumps(game['tags'], ensure_ascii=False)}
"""

    response = dashscope.MultiModalConversation.call(
        api_key=DASHSCOPE_API_KEY,
        model=LLM_MODEL,
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": [
                    {
                        "text": USER_PROMPT
                    }
                ]
            }
        ]
    )

    return (
        response.output
        .choices[0]
        .message.content[0]["text"]
    )


def explain_item_similarity(
    source_game,
    recommended_game
):

    SYSTEM_PROMPT = """
You are a game recommendation explanation assistant.

Explain why the recommended game is similar to the source game.

Rules:
- Maximum 80 words.
"""

    USER_PROMPT = f"""
Source Game:
{source_game['name']}

Source Summary:
{source_game['summary']}

Source Tags:
{json.dumps(source_game['tags'], ensure_ascii=False)}

Recommended Game:
{recommended_game['name']}

Recommended Summary:
{recommended_game['summary']}

Recommended Tags:
{json.dumps(recommended_game['tags'], ensure_ascii=False)}
"""

    response = dashscope.MultiModalConversation.call(
        api_key=DASHSCOPE_API_KEY,
        model=LLM_MODEL,
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": [
                    {
                        "text": USER_PROMPT
                    }
                ]
            }
        ]
    )

    return (
        response.output
        .choices[0]
        .message.content[0]["text"]
    )

## Only retrievals (#4)

In [52]:
# ============================================================
# HYBRID RETRIEVAL
# ============================================================
# Free-text Query -> Top-N Games
# Keyword Search + Vector Search + RRF Fusion

from collections import defaultdict

print("\n")
print("="*80)
print("HYBRID RETRIEVAL")
print("="*80)

#query = "relaxing farming game"
#query = "a relaxing video game with beautiful art but no combat"
query = "a truck driving game featured in europe"
#query = "A Game About Chopping Trees"

top_n = 10

print(f"Query: {query}")
print(f"top_n: {top_n}")

keyword_results = keyword_search(
    query,
    top_k=50
)

vector_results = vector_search(
    query,
    top_k=50
)

scores = defaultdict(float)

for rank, (game, _) in enumerate(
    keyword_results,
    start=1
):
    scores[game["item_id"]] += (
        1 / (60 + rank)
    )

for rank, (game, _) in enumerate(
    vector_results,
    start=1
):
    scores[game["item_id"]] += (
        1 / (60 + rank)
    )

game_lookup = {}

for game, _ in keyword_results:
    game_lookup[game["item_id"]] = game

for game, _ in vector_results:
    game_lookup[game["item_id"]] = game

final_results = sorted(
    scores.items(),
    key=lambda x: x[1],
    reverse=True
)[:top_n]

for rank, (item_id, score) in enumerate(
    final_results,
    start=1
):

    print(f"\nRank #{rank}")

    print_game(
        game_lookup[item_id],
        score
    )



HYBRID RETRIEVAL
Query: a truck driving game featured in europe
top_n: 10


/var/folders/dn/8xt3xqsn3ysf586z6l0zvfym0000gn/T/ipykernel_4440/2174385808.py:78: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(



Rank #1
--------------------------------------------------------------------------------
Euro Truck Simulator 2 (id=227300) score=0.0323
Mood: Relaxing, Contemplative
Theme: Modern
Summary: The player drives a truck across more than 60 European cities, hauling varied cargo over real road networks while building a fleet, buying garages, hiring drivers, and managing a g

Rank #2
--------------------------------------------------------------------------------
Transport Fever 2 (id=1066780) score=0.0164
Mood: Relaxing, Contemplative
Theme: Historical, Modern
Summary: A transport simulation game where the player builds rail, road, water and air networks across three continents from 1850 to the present, moving passengers and goods to grow cities 

Rank #3
--------------------------------------------------------------------------------
Insane 2 (id=35320) score=0.0161
Mood: Challenging, Epic
Theme: Nature, Modern
Summary: An off-road racing game offering ten modes including capture the flag,

/var/folders/dn/8xt3xqsn3ysf586z6l0zvfym0000gn/T/ipykernel_4440/2174385808.py:165: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


In [57]:
# ============================================================
# HYBRID ITEM SIMILARITY
# ============================================================
# Game ID -> Top-N Similar Games
# Keyword Similarity + Vector Similarity + RRF Fusion

print("\n")
print("="*80)
print("HYBRID ITEM SIMILARITY")
print("="*80)

target_id = 227300 # ETS2
#target_id = 413150 # Stardew Valley

top_n = 10

print(f"target_id: {target_id}")
print(f"top_n: {top_n}")

target = pd.read_sql(
    """
    SELECT
        item_id,
        name
    FROM items
    WHERE item_id = %s
    """,
    conn,
    params=(target_id,)
).iloc[0]

print(
    f"\nTarget item: "
    f"{target['name']} "
    f"({target_id})"
)

results = hybrid_similar_db(
    target_id,
    top_k=top_n
)

for rank, (game, score) in enumerate(
    results,
    start=1
):

    print(f"\nRank #{rank}")

    print_game(
        game,
        score
    )



HYBRID ITEM SIMILARITY
target_id: 227300
top_n: 10

Target item: Euro Truck Simulator 2 (227300)


/var/folders/dn/8xt3xqsn3ysf586z6l0zvfym0000gn/T/ipykernel_4440/1063534808.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  target = pd.read_sql(
/var/folders/dn/8xt3xqsn3ysf586z6l0zvfym0000gn/T/ipykernel_4440/2174733378.py:202: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  target = pd.read_sql(
/var/folders/dn/8xt3xqsn3ysf586z6l0zvfym0000gn/T/ipykernel_4440/2174733378.py:78: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(



Rank #1
--------------------------------------------------------------------------------
Truck & Logistics Simulator (id=873840) score=0.0164
Mood: Relaxing, Challenging
Theme: Modern
Summary: A delivery simulation where the player loads pallets, containers, and vehicles with forklifts and handlers, then drives over 30 customizable vehicles across a 28 km² open world, co

Rank #2
--------------------------------------------------------------------------------
Portal (id=400) score=0.0161
Mood: Mysterious, Atmospheric
Theme: Sci-Fi, Mystery
Summary: Players navigate the mysterious Aperture Science Laboratories, solving physical puzzles by opening portals to move objects and themselves through space. The single-player game chal

Rank #3
--------------------------------------------------------------------------------
Truck Driver (id=768180) score=0.0161
Mood: Relaxing, Cheerful
Theme: Modern
Summary: After inheriting a truck from his father, the player moves to a new city and builds a t

/var/folders/dn/8xt3xqsn3ysf586z6l0zvfym0000gn/T/ipykernel_4440/666004318.py:150: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


## Retrievals + LLM explains (#5)

In [3]:
# ============================================================
# HYBRID RETRIEVAL + LLM EXPLANATION
# ============================================================
# Free-text Query -> Top-N Games
# Keyword Search + Vector Search + RRF Fusion

from collections import defaultdict

print("\n")
print("="*80)
print("HYBRID RETRIEVAL + LLM EXPLANATION")
print("="*80)

#query = "relaxing farming game"
#query = "A Game About Chopping Trees"
#query = "a truck driving game featured in europe"
query = "a relaxing video game with beautiful art but no combat"
#query = "truck driving game"

top_n = 10

print(f"Query: {query}")
print(f"top_n: {top_n}")

keyword_results = keyword_search(
    query,
    top_k=50
)

vector_results = vector_search(
    query,
    top_k=50
)

scores = defaultdict(float)

for rank, (game, _) in enumerate(
    keyword_results,
    start=1
):
    scores[game["item_id"]] += (
        1 / (60 + rank)
    )

for rank, (game, _) in enumerate(
    vector_results,
    start=1
):
    scores[game["item_id"]] += (
        1 / (60 + rank)
    )

game_lookup = {}

for game, _ in keyword_results:
    game_lookup[game["item_id"]] = game

for game, _ in vector_results:
    game_lookup[game["item_id"]] = game

final_results = sorted(
    scores.items(),
    key=lambda x: x[1],
    reverse=True
)[:top_n]

for rank, (item_id, score) in enumerate(
    final_results,
    start=1
):

    print(f"\nRank #{rank}")

    print_game(
        game_lookup[item_id],
        score
    )

top_game = game_lookup[
    final_results[0][0]
]

print("\n")
print("LLM Explanation")
print("-"*80)

print(
    explain_query_match(
        query,
        top_game
    )
)



HYBRID RETRIEVAL + LLM EXPLANATION
Query: a relaxing video game with beautiful art but no combat
top_n: 10


/var/folders/dn/8xt3xqsn3ysf586z6l0zvfym0000gn/T/ipykernel_7643/2292832092.py:100: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(
/var/folders/dn/8xt3xqsn3ysf586z6l0zvfym0000gn/T/ipykernel_7643/2292832092.py:188: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(



Rank #1
--------------------------------------------------------------------------------
Pleasure Puzzle:Portrait 趣拼拼：肖像画 (id=939620) score=0.0164
Mood: Relaxing, Cozy
Theme: Other
Summary: A classic jigsaw puzzle game built from one hundred authorized portrait images, with custom difficulty on the first hundred puzzles. Players assemble pieces while listening to back

Rank #2
--------------------------------------------------------------------------------
Relaxing Simulator (id=2965380) score=0.0164
Mood: Relaxing, Cozy, Contemplative
Theme: Nature, Surreal
Summary: A stress-free experience where the player becomes a floating spirit and drifts through peaceful landscapes alone or with friends. There are no objectives or story; players explore 

Rank #3
--------------------------------------------------------------------------------
Submerged: Hidden Depths (id=1614270) score=0.0161
Mood: Relaxing, Atmospheric, Melancholic
Theme: Post-Apocalyptic, Mystery
Summary: Players take on the 

In [4]:
# ============================================================
# HYBRID ITEM SIMILARITY + LLM EXPLANATION
# ============================================================
# Game ID -> Top-N Similar Games
# Keyword Similarity + Vector Similarity + RRF Fusion

print("\n")
print("="*80)
print("HYBRID ITEM SIMILARITY + LLM EXPLANATION")
print("="*80)

target_id = 227300 # ETS2
#target_id = 413150 # Stardew Valley

top_n = 10

print(f"target_id: {target_id}")
print(f"top_n: {top_n}")

source = pd.read_sql(
    """
    SELECT
        i.item_id,
        i.name,
        a.summary,
        a.tags
    FROM items i
    JOIN item_annotations a
        USING(item_id)
    WHERE i.item_id = %s
    """,
    conn,
    params=(target_id,)
).iloc[0]

print(
    f"\nTarget item: "
    f"{source['name']} "
    f"({target_id})"
)

results = hybrid_similar_db(
    target_id,
    top_k=top_n
)

for rank, (game, score) in enumerate(
    results,
    start=1
):

    print(f"\nRank #{rank}")

    print_game(
        game,
        score
    )

source_game = {
    "item_id": source["item_id"],
    "name": source["name"],
    "summary": source["summary"],
    "tags": source["tags"]
}

recommended_game = results[0][0]

print("\n")
print("LLM Explanation")
print("-"*80)

print(
    explain_item_similarity(
        source_game,
        recommended_game
    )
)




HYBRID ITEM SIMILARITY + LLM EXPLANATION
target_id: 227300
top_n: 10

Target item: Euro Truck Simulator 2 (227300)


/var/folders/dn/8xt3xqsn3ysf586z6l0zvfym0000gn/T/ipykernel_7643/2233770173.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  source = pd.read_sql(
/var/folders/dn/8xt3xqsn3ysf586z6l0zvfym0000gn/T/ipykernel_7643/2292832092.py:304: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  target = pd.read_sql(
/var/folders/dn/8xt3xqsn3ysf586z6l0zvfym0000gn/T/ipykernel_7643/2292832092.py:100: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(
/var/folders/dn/8xt3xqsn3ysf586z6l0zvfym0000gn/T/ipykernel_7643/2292832092.py:266


Rank #1
--------------------------------------------------------------------------------
Truck & Logistics Simulator (id=873840) score=0.0164
Mood: Relaxing, Challenging
Theme: Modern
Summary: A delivery simulation where the player loads pallets, containers, and vehicles with forklifts and handlers, then drives over 30 customizable vehicles across a 28 km² open world, co

Rank #2
--------------------------------------------------------------------------------
Two Worlds Epic Edition (id=1930) score=0.0161
Mood: Epic, Mysterious
Theme: Fantasy, Medieval
Summary: A bounty hunter and mercenary searches for his missing sister Kyra, whose disappearance is tied to a family relict and a dark Brotherhood. In the fantasy land of Antaloor, the play

Rank #3
--------------------------------------------------------------------------------
Truck Driver (id=768180) score=0.0161
Mood: Relaxing, Cheerful
Theme: Modern
Summary: After inheriting a truck from his father, the player moves to a new city a